# Discover constructs and build a codebook

This tutorial starts with documents and a theoretical framework, rather than a
predefined codebook. It discovers constructs, consolidates their names, lets you
review and edit the groups, and builds a codebook for the coding pipeline.

**Requires the updated discovery module supplied with the discovery refactor.**
If you replaced the module while a notebook was open, restart the kernel first.
Use the Python environment where `llm_tracker` and Jupyter are installed.

Discovery and semantic merging make LLM calls. Manual edits, filtering, formatting,
and codebook creation run locally. An optional reevaluation step makes another
LLM call. No API key is stored in this notebook.


## 1. Set paths and options

Run the demo first, or set `use_demo_data=False` and point `input_path` to your data.
Relative paths are resolved from the notebook's working directory.

- TXT: one document per file; set `text_column=None` and `id_column=None`.
- CSV/XLSX/XLS: set `text_column` for one document per row; `id_column` is optional.
- A directory is read nonrecursively. The column settings apply to every tabular file.

For tabular data, omitting `text_column` treats the entire table as one document.
Set `model_name` to an available model ID for your provider; leaving it `None`
uses the package's configured default. Set `OPENROUTER_API_KEY` in your environment
or specify a local `.env` file. Never commit that file or print your key.


In [46]:
from pathlib import Path

use_demo_data = False

framework = "Cognitive behavioral therapy"  # Any framework name is accepted.

input_path = Path(
   
)
output_root = Path(
    
)

text_column = "post"
id_column = None  # Uses the row number as each document's ID.

provider = "openrouter"
model_name = "google/gemini-3.1-pro-preview" # Or your chosen provider model ID.
env_file = "" # Or Path("/absolute/path/to/.env")
max_retries = 2

fuzzy_threshold = 0.9  # None disables fuzzy spelling matching.
seed = 0
run_reevaluation = False  # True makes an additional LLM call after filtering.
n_codebook_examples = 3


In [31]:
import json
from copy import deepcopy

import pandas as pd
from IPython.display import display

import llm_tracker.discovery as discovery_module
from llm_tracker.config import AnalyzerConfig
from llm_tracker.discovery_v2 import (
    DISCOVERY_PROMPT,
    MERGE_PROMPT,
    REEVALUATE_PROMPT,
    LLMTrackerDiscoverer,
    apply_merge_edits,
    build_codebook,
    codebook_to_frame,
    collapse_variants,
    filter_merged_constructs,
    format_merged_constructs,
    frame_to_codebook,
    merge_constructs,
    reevaluate_merged_constructs,
    save_codebook,
)

print("Discovery module:", discovery_module.__file__)
print("Working directory:", Path.cwd())


Discovery module: /Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/discovery.py
Working directory: /Users/freymon.perez/Projects/GitHub/llm_tracker


The demo cell creates a CSV only when `use_demo_data=True`. Change the input path
before using your own data. An existing demo file is left untouched.


In [32]:
if use_demo_data:
    text_column, id_column = "text", "doc_id"
    input_path.parent.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        demo = pd.DataFrame([
            {"doc_id": "demo_1", "text": "If I make one mistake at work, everyone will think I am incompetent."},
            {"doc_id": "demo_2", "text": "I assume my friend is angry whenever she takes a while to reply."},
            {"doc_id": "demo_3", "text": "I felt anxious about the meeting, so I avoided opening the invitation."},
            {"doc_id": "demo_4", "text": "I wrote down the evidence for and against my fear, which helped me consider another explanation."},
            {"doc_id": "demo_5", "text": "I took one small step toward the task instead of putting it off again."},
            {"doc_id": "demo_6", "text": "The delivery contained three blue folders and a box of paper clips."},
        ])
        demo.to_csv(input_path, index=False)
    display(pd.read_csv(input_path))

if not input_path.exists():
    raise FileNotFoundError(f"Update input_path; no input exists at {input_path.resolve()}")

discovery_dir = output_root / "discovery"
merge_dir = output_root / "merged"
edits_dir = output_root / "edited"
filtered_dir = output_root / "filtered"
reevaluation_dir = output_root / "reevaluated"
codebook_dir = output_root / "codebooks"
output_root.mkdir(parents=True, exist_ok=True)


## 2. Configure the discoverer and inspect its prompt

The discovery prompt uses `{framework}` and `{document}`. Keep these placeholders
when customizing it. Literal JSON braces in a template must be doubled: `{{` and `}}`.
Printing a template does not call the API.


In [33]:
config_kwargs = {
    "provider": provider,
    "max_retries": max_retries,
    "temperature": 0.0,
}
if model_name is not None:
    config_kwargs["model_name"] = model_name
if env_file is not None:
    config_kwargs["api_key"] = str(env_file)

config = AnalyzerConfig(**config_kwargs)
discoverer = LLMTrackerDiscoverer(config=config)
print("Model:", config.model_name)
print(discoverer.discovery_prompt)


Model: google/gemini-3.1-pro-preview
You are an expert qualitative researcher analyzing a document through the lens of a specific theoretical framework.

Framework: {framework}

Read the document below and identify the constructs from this framework that are present in the text. Use your own judgement about what counts as a construct of this framework; you are not working from a fixed list.

For each construct you find, provide:
- "construct": a short name for the construct
- "description": one sentence describing the construct as it appears here
- "quotes": a list of verbatim quotes from the document that evidence it

Only report constructs genuinely grounded in the text. If none are present, return an empty list.
Spell out construct names and descriptions in full. Do not use acronyms such as
CBT; write cognitive behavioral therapy instead.

Respond ONLY with a JSON object of the form:
{{"constructs": [{{"construct": "...", "description": "...", "quotes": ["..."]}}]}}

Document:
{docu

In [34]:
# Demonstrate changing the prompt, then restore the default before discovery.
discoverer.discovery_prompt = DISCOVERY_PROMPT + "\nUse concise construct names and definitions."
print(discoverer.discovery_prompt)

discoverer.discovery_prompt = DISCOVERY_PROMPT
assert discoverer.discovery_prompt == DISCOVERY_PROMPT


You are an expert qualitative researcher analyzing a document through the lens of a specific theoretical framework.

Framework: {framework}

Read the document below and identify the constructs from this framework that are present in the text. Use your own judgement about what counts as a construct of this framework; you are not working from a fixed list.

For each construct you find, provide:
- "construct": a short name for the construct
- "description": one sentence describing the construct as it appears here
- "quotes": a list of verbatim quotes from the document that evidence it

Only report constructs genuinely grounded in the text. If none are present, return an empty list.
Spell out construct names and descriptions in full. Do not use acronyms such as
CBT; write cognitive behavioral therapy instead.

Respond ONLY with a JSON object of the form:
{{"constructs": [{{"construct": "...", "description": "...", "quotes": ["..."]}}]}}

Document:
{document}

Use concise construct names an

## 3. Discover constructs in the documents

This cell makes LLM calls: normally one per document, with retries as configured.
Rerunning it sends every input document again. Later steps use the dataframe in
memory and do not repeat discovery.

The result has one row per supporting quote. A detected construct with no quote
gets an empty-quote row. Successful documents with no detected constructs have no
rows, but still count toward the frequency denominator. Failed documents do not.
The tutorial stops on failures so you can review them before interpreting results.


In [35]:
discoveries, discovery_metadata, discovery_errors = discoverer.discover(
    path=input_path,
    framework=framework,
    text_column=text_column,
    id_column=id_column,
    output_dir=discovery_dir,
    save_format="both",
)

print("Successfully processed documents:", discoveries.attrs["total_documents"])
print("Discovery rows:", len(discoveries))
print("Failures:", len(discovery_errors))
display(discoveries)

if discovery_errors:
    display(pd.DataFrame([error.model_dump() for error in discovery_errors]))
    raise RuntimeError("Review the saved discovery errors before continuing. Rerunning discovery processes every document again.")
if discoveries.empty:
    raise RuntimeError("No constructs were discovered. Review the inputs, framework, and prompt before merging.")


Discovering across 60 document(s) from 1 file(s).

Discovering [1/60]: reddit_autism_anxiety_depression.csv::0
  OK: 5 construct(s), 7 quote(s)
Discovering [2/60]: reddit_autism_anxiety_depression.csv::1
  OK: 5 construct(s), 8 quote(s)
Discovering [3/60]: reddit_autism_anxiety_depression.csv::2
  OK: 4 construct(s), 6 quote(s)
Discovering [4/60]: reddit_autism_anxiety_depression.csv::3
  OK: 5 construct(s), 7 quote(s)
Discovering [5/60]: reddit_autism_anxiety_depression.csv::4
  OK: 3 construct(s), 5 quote(s)
Discovering [6/60]: reddit_autism_anxiety_depression.csv::5
  OK: 3 construct(s), 3 quote(s)
Discovering [7/60]: reddit_autism_anxiety_depression.csv::6
  OK: 4 construct(s), 4 quote(s)
Discovering [8/60]: reddit_autism_anxiety_depression.csv::7
  OK: 4 construct(s), 4 quote(s)
Discovering [9/60]: reddit_autism_anxiety_depression.csv::8
  OK: 4 construct(s), 9 quote(s)
Discovering [10/60]: reddit_autism_anxiety_depression.csv::9
  OK: 4 construct(s), 4 quote(s)
Discovering [11/60

,filename,doc_id,construct,description,quote
0,reddit_autism_anxiety_depression.csv,0,Negative Automatic Thoughts,"The individual experiences spontaneous, negati...","And i feel like saying watching tv, reading, ..."
1,reddit_autism_anxiety_depression.csv,0,Behavioral Avoidance,The individual describes a history and ongoing...,I know my anxiety has made it hard for me to g...
2,reddit_autism_anxiety_depression.csv,0,Behavioral Avoidance,The individual describes a history and ongoing...,Because again i know anxiety can make it hard ...
3,reddit_autism_anxiety_depression.csv,0,Behavioral Avoidance,The individual describes a history and ongoing...,I'm not a interesting person and as shitty as ...
4,reddit_autism_anxiety_depression.csv,0,Core Beliefs,"The individual expresses a fundamental, deeply...",I'm not a interesting person
...,...,...,...,...,...
341,reddit_autism_anxiety_depression.csv,59,Maladaptive Coping Behaviors,"The individual engages in harmful behaviors, s...",started to smoke and drink because of it.
342,reddit_autism_anxiety_depression.csv,59,Maladaptive Coping Behaviors,"The individual engages in harmful behaviors, s...",I get drunk usually every other day now and cr...
343,reddit_autism_anxiety_depression.csv,59,Hopelessness,The individual expresses a profound negative e...,i dont know how i should go on in life anymore.
344,reddit_autism_anxiety_depression.csv,59,Hopelessness,The individual expresses a profound negative e...,And with no close family members this life see...


## 4. Inspect name normalization before semantic merging

This step protects against small variations in naming from being interrpreted as entirely different constructs.

This step involves no LLM calls. Case, separators, and extra spaces are normalized; constituent
names become lowercase. Optional fuzzy matching will also match near spellsings, 
please note this is based on spelling only, not meaning. To avoid this simply set
`fuzzy_threshold=None` to use the most basic name normalization.

`count` in this intermediate table counts distinct `(filename, doc_id)` pairs.
`instance_count` counts nonempty quote occurrences before deduplication. The final
merge table calls the document count `document_count`.


In [36]:
normalized = collapse_variants(discoveries, fuzzy_threshold=fuzzy_threshold)
display(normalized[["construct", "count", "instance_count", "variants"]])
display(pd.DataFrame(normalized.attrs.get("name_matches", [])))


,construct,count,instance_count,variants
0,negative automatic thoughts,11,14,[Negative Automatic Thoughts]
1,activating event,8,12,"[Activating Event, Activating Events]"
2,cognitive distortions,6,9,"[Cognitive Distortion, Cognitive Distortions, ..."
3,overgeneralization,6,6,[Overgeneralization]
4,fortune telling,6,10,"[Fortune Telling, Fortune telling]"
...,...,...,...,...
118,cost benefit analysis,1,2,[Cost-Benefit Analysis]
119,core beliefs of defectiveness,1,3,[Core Beliefs of Defectiveness]
120,core beliefs about others,1,2,[Core Beliefs About Others]
121,core belief of incompetence,1,3,[Core Belief of Incompetence]


,original_name,construct_found
0,"""Should"" Statements",should statements
1,Acceptance,acceptance
2,Activating Event,activating event
3,Activating Events,activating event
4,Adaptive Coping Strategies,adaptive coping strategies
...,...,...
131,Should Statements,should statements
132,Social Comparison,social comparison
133,Somatic Symptoms of Anxiety,somatic symptoms of anxiety
134,Suicidal Intent and Planning,suicidal intent and planning


## 5. Merge constructs by meaning

Merging makes an LLM call to group related constituents, write merged names and
definitions, and assign prototypicality scores. Inspect or customize this template
using `{framework}`, `{constructs}`, and `{existing_note}`; the example restores
the default before running.


In [37]:
print(MERGE_PROMPT)
merge_prompt = MERGE_PROMPT + "\nPrefer concise, descriptive group names."
print(merge_prompt)
merge_prompt = MERGE_PROMPT


You are an expert qualitative researcher consolidating constructs discovered across many documents under the framework: {framework}.

Below is a list of discovered constructs, each with a description. They were named freely by a model reading one document at a time, so the same idea often appears under different names. Your task is to group them into canonical constructs.

For each canonical construct provide:
- "name": the canonical name (synthesize the best name; it may be one of the grouped names)
- "definition": a one-sentence definition of the consolidated construct
- "constituents": the discovered constructs you grouped into it, each with a "prototypicality" score from 0 to 1 (1 = the first thing that comes to mind for this canonical construct; lower = related but less obvious)

Guidance:
- Group on meaning, using the descriptions as well as the names.
- A construct does NOT have to be merged with anything. If it stands on its own, return it as a canonical construct with an empty

In [38]:
merged, merge_table = merge_constructs(
    df=discoveries,
    config=config,
    framework=framework,
    merge_prompt=merge_prompt,
    fuzzy_threshold=fuzzy_threshold,
    seed=seed,
    batch=False,
    max_prompt_chars=60000,
    output_dir=merge_dir,
    save_format="both",
)

display(format_merged_constructs(merge_table, n_quotes=3, seed=seed))


Collapsed 136 discovered name(s) into 123 distinct construct(s).
Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs_test_reddit/merged


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Acceptance,A cognitive and behavioral stance of acknowled...,merged,acceptance,1,1,1.666667,60,1.0,We generally accepting and just accept it,1.7% (N=60)
1,Activating Event,"External situations, environmental stressors, ...",merged,activating event,8,12,13.333333,60,1.0,"On top of that nearly every night, around the ...",13.3% (N=60)
2,Activating Event,"External situations, environmental stressors, ...",merged,environmental stressors,1,1,1.666667,60,0.8,I’ve had a lot of trauma and bad things happen...,1.7% (N=60)
3,Activating Event,"External situations, environmental stressors, ...",merged,behavioral antecedent,1,2,1.666667,60,0.8,For him the see a bit of a positive ( boredem)...,1.7% (N=60)
4,Activating Event,"External situations, environmental stressors, ...",merged,behavioral triggers,1,2,1.666667,60,0.8,Problem is that one of my most 'favorite' ways...,1.7% (N=60)
...,...,...,...,...,...,...,...,...,...,...,...
118,Self-Monitoring,Metacognitive awareness and observation of one...,merged,self monitoring,1,1,1.666667,60,1.0,I don't know how best to describe my feelings ...,1.7% (N=60)
119,Self-Monitoring,Metacognitive awareness and observation of one...,merged,mood monitoring,1,2,1.666667,60,0.9,Feeling better for the 1st time in a whild;\nt...,1.7% (N=60)
120,Should Statements,A cognitive distortion involving the use of ri...,merged,should statements,4,7,6.666667,60,1.0,So many of my friends have left; and the fact ...,6.7% (N=60)
121,Social Comparison,Evaluating oneself negatively by comparing one...,merged,social comparison,1,1,1.666667,60,1.0,All my friends have a gf and they are getting ...,1.7% (N=60)


Each row describes one `construct_found` within a `merged_construct`.

| Column | Meaning |
|---|---|
| `document_count` | Distinct documents containing this constituent |
| `instance_count` | Nonempty quote occurrences, before duplicate quote text is removed |
| `document_frequency_percent` | `100 × document_count / total_documents` |
| `total_documents` | All successful documents, including those with no discoveries |
| `prototypicality` | LLM score from 0 to 1 for how representative the constituent is |
| `status` | `merged`, `standalone`, or `unplaced` |

The formatted view adds `document_frequency`, for example `20.0% (N=10)`.
The denominator is not the number of positive documents. A standalone construct
was kept independent; an unplaced construct was omitted by the merging response
and rescued into its own group. Missing scores remain missing.

Use the **raw `merge_table`** for later functions. The formatted view joins quotes
into display strings. For large inputs, `batch=True` splits merging into sequential
calls; this is different from the provider's asynchronous Batch API.


## 6. Review and apply manual grouping edits

The dictionary below is also saved as `merged_constructs_groups_<timestamp>.json`.
You can rename a key, move a constituent between lists, or omit a constituent to
remove it. Each known constituent may appear only once. This function makes no
LLM calls. For direct Python pasting, the following values contain only strings
and lists, so the printed JSON also works as a Python dictionary.


In [39]:
grouping = {
    name: group["construct_found"].tolist()
    for name, group in merge_table.groupby("merged_construct", sort=False)
}
print(json.dumps(grouping, indent=2, ensure_ascii=False))


{
  "Acceptance": [
    "acceptance"
  ],
  "Activating Event": [
    "activating event",
    "environmental stressors",
    "behavioral antecedent",
    "behavioral triggers"
  ],
  "Adaptive Coping Strategies": [
    "adaptive coping strategies",
    "coping strategies",
    "coping strategy",
    "coping behavior",
    "behavioral coping strategy",
    "coping skill seeking",
    "compensatory coping strategy"
  ],
  "All-or-Nothing Thinking": [
    "all or nothing thinking"
  ],
  "Anhedonia and Emotional Numbing": [
    "anhedonia",
    "anhedonia and emotional numbing",
    "emotional numbing"
  ],
  "Anxiety": [
    "anxiety",
    "anticipatory anxiety",
    "emotional consequence anxiety",
    "fear of negative evaluation"
  ],
  "Avoidance Behavior": [
    "avoidance behavior",
    "behavioral avoidance",
    "avoidance coping",
    "escape avoidance",
    "behavioral withdrawal",
    "behavioral isolation",
    "maladaptive behavioral response"
  ],
  "Beck's Cognitive Triad"

In [42]:
# Replace this assignment with your edited dictionary, or edit the copy below.
edited_grouping = deepcopy(grouping)

# Examples, using names that actually occur in the printed dictionary:
# edited_grouping["New group name"] = edited_grouping.pop("Old group name")
# edited_grouping["Group name"].remove("constituent to remove")

edited_merged, edited_table = apply_merge_edits(
    table=merge_table,
    grouping=edited_grouping,
    # definitions={"New group name": "Your revised definition."},
    output_dir=edits_dir,
    save_format="both",
)
display(format_merged_constructs(edited_table, n_quotes=3, seed=seed))


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs_test_reddit/edited


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Acceptance,A cognitive and behavioral stance of acknowled...,merged,acceptance,1,1,1.666667,60,1.0,We generally accepting and just accept it,1.7% (N=60)
1,Activating Event,"External situations, environmental stressors, ...",merged,activating event,8,12,13.333333,60,1.0,"On top of that nearly every night, around the ...",13.3% (N=60)
2,Activating Event,"External situations, environmental stressors, ...",merged,environmental stressors,1,1,1.666667,60,0.8,I’ve had a lot of trauma and bad things happen...,1.7% (N=60)
3,Activating Event,"External situations, environmental stressors, ...",merged,behavioral antecedent,1,2,1.666667,60,0.8,For him the see a bit of a positive ( boredem)...,1.7% (N=60)
4,Activating Event,"External situations, environmental stressors, ...",merged,behavioral triggers,1,2,1.666667,60,0.8,Problem is that one of my most 'favorite' ways...,1.7% (N=60)
...,...,...,...,...,...,...,...,...,...,...,...
118,Self-Monitoring,Metacognitive awareness and observation of one...,merged,self monitoring,1,1,1.666667,60,1.0,I don't know how best to describe my feelings ...,1.7% (N=60)
119,Self-Monitoring,Metacognitive awareness and observation of one...,merged,mood monitoring,1,2,1.666667,60,0.9,Feeling better for the 1st time in a whild;\nt...,1.7% (N=60)
120,Should Statements,A cognitive distortion involving the use of ri...,merged,should statements,4,7,6.666667,60,1.0,So many of my friends have left; and the fact ...,6.7% (N=60)
121,Social Comparison,Evaluating oneself negatively by comparing one...,merged,social comparison,1,1,1.666667,60,1.0,All my friends have a gf and they are getting ...,1.7% (N=60)


Counts and quotes stay with their constituents. The denominator stays fixed.
Existing definitions are preserved where a source is identifiable; a new mixed
group may need an explicit definition. Moved members lose their old prototypicality
score because it described their previous group. A pure group rename retains
scores. Definitions can be outdated after membership changes; reevaluation below
can update names and definitions, but does not rescore prototypicality.


## 7. Filter the merge results

This is also local. Minimum thresholds are inclusive, and all supplied criteria
must pass. Removal names are matched using case/separator normalization. Unknown
removal names raise an error.

If `min_prototypicality` is supplied, missing scores are excluded—including
standalone/unplaced constructs and moved members. Leave it `None` to retain them.
Filtering never reduces the successful-document denominator.


In [43]:
remove_constructs = []       # Constituent names to remove.
remove_merged_constructs = []  # Whole merged groups to remove.
min_document_count = 1
min_instance_count = 1
min_prototypicality = None

filtered_merged, filtered_table = filter_merged_constructs(
    table=edited_table,
    remove_constructs=remove_constructs,
    remove_merged_constructs=remove_merged_constructs,
    min_document_count=min_document_count,
    min_instance_count=min_instance_count,
    min_prototypicality=min_prototypicality,
    output_dir=filtered_dir,
    save_format="both",
)
display(format_merged_constructs(filtered_table, n_quotes=3, seed=seed))
print("Constituents remaining:", len(filtered_table))
print("Removed members recorded:", len(filtered_table.attrs.get("removed_constituents", [])))

if filtered_table.empty:
    raise RuntimeError("All constituents were removed. Relax the filters or revise the grouping before building a codebook.")


Saved merge output to /Users/freymon.perez/Projects/GitHub/llm_tracker/discovery_outputs_test_reddit/filtered


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Acceptance,A cognitive and behavioral stance of acknowled...,merged,acceptance,1,1,1.666667,60,1.0,We generally accepting and just accept it,1.7% (N=60)
1,Activating Event,"External situations, environmental stressors, ...",merged,activating event,8,12,13.333333,60,1.0,"On top of that nearly every night, around the ...",13.3% (N=60)
2,Activating Event,"External situations, environmental stressors, ...",merged,environmental stressors,1,1,1.666667,60,0.8,I’ve had a lot of trauma and bad things happen...,1.7% (N=60)
3,Activating Event,"External situations, environmental stressors, ...",merged,behavioral antecedent,1,2,1.666667,60,0.8,For him the see a bit of a positive ( boredem)...,1.7% (N=60)
4,Activating Event,"External situations, environmental stressors, ...",merged,behavioral triggers,1,2,1.666667,60,0.8,Problem is that one of my most 'favorite' ways...,1.7% (N=60)
...,...,...,...,...,...,...,...,...,...,...,...
118,Self-Monitoring,Metacognitive awareness and observation of one...,merged,self monitoring,1,1,1.666667,60,1.0,I don't know how best to describe my feelings ...,1.7% (N=60)
119,Self-Monitoring,Metacognitive awareness and observation of one...,merged,mood monitoring,1,2,1.666667,60,0.9,Feeling better for the 1st time in a whild;\nt...,1.7% (N=60)
120,Should Statements,A cognitive distortion involving the use of ri...,merged,should statements,4,7,6.666667,60,1.0,So many of my friends have left; and the fact ...,6.7% (N=60)
121,Social Comparison,Evaluating oneself negatively by comparing one...,merged,social comparison,1,1,1.666667,60,1.0,All my friends have a gf and they are getting ...,1.7% (N=60)


Constituents remaining: 123
Removed members recorded: 0


## 8. Optionally reevaluate merged names and definitions

Set `run_reevaluation=True` in the options cell to enable this extra LLM call.
The model receives the previous groups, current groups, and removed constituents.
It may revise names and definitions only; it cannot move or restore constituents.
Quotes, counts, the denominator, and existing/missing scores stay unchanged.

The template is available as `REEVALUATE_PROMPT`. A custom template uses
`{framework}`, `{previous}`, `{removed}`, and `{current}` and must keep the response
schema. The full prompt is checked against `max_prompt_chars` before sending.


In [44]:
# Print this to inspect the reevaluation instructions without an API call:
# print(REEVALUATE_PROMPT)

if run_reevaluation:
    final_merged, final_table = reevaluate_merged_constructs(
        table=filtered_table,
        config=config,
        framework=framework,
        reevaluation_prompt=REEVALUATE_PROMPT,
        max_prompt_chars=60000,
        output_dir=reevaluation_dir,
        save_format="both",
    )
else:
    final_merged, final_table = filtered_merged, filtered_table
    print("Reevaluation skipped; using the locally reviewed and filtered result.")

display(format_merged_constructs(final_table, n_quotes=3, seed=seed))


Reevaluation skipped; using the locally reviewed and filtered result.


,merged_construct,definition,status,construct_found,document_count,instance_count,document_frequency_percent,total_documents,prototypicality,quotes,document_frequency
0,Acceptance,A cognitive and behavioral stance of acknowled...,merged,acceptance,1,1,1.666667,60,1.0,We generally accepting and just accept it,1.7% (N=60)
1,Activating Event,"External situations, environmental stressors, ...",merged,activating event,8,12,13.333333,60,1.0,"On top of that nearly every night, around the ...",13.3% (N=60)
2,Activating Event,"External situations, environmental stressors, ...",merged,environmental stressors,1,1,1.666667,60,0.8,I’ve had a lot of trauma and bad things happen...,1.7% (N=60)
3,Activating Event,"External situations, environmental stressors, ...",merged,behavioral antecedent,1,2,1.666667,60,0.8,For him the see a bit of a positive ( boredem)...,1.7% (N=60)
4,Activating Event,"External situations, environmental stressors, ...",merged,behavioral triggers,1,2,1.666667,60,0.8,Problem is that one of my most 'favorite' ways...,1.7% (N=60)
...,...,...,...,...,...,...,...,...,...,...,...
118,Self-Monitoring,Metacognitive awareness and observation of one...,merged,self monitoring,1,1,1.666667,60,1.0,I don't know how best to describe my feelings ...,1.7% (N=60)
119,Self-Monitoring,Metacognitive awareness and observation of one...,merged,mood monitoring,1,2,1.666667,60,0.9,Feeling better for the 1st time in a whild;\nt...,1.7% (N=60)
120,Should Statements,A cognitive distortion involving the use of ri...,merged,should statements,4,7,6.666667,60,1.0,So many of my friends have left; and the fact ...,6.7% (N=60)
121,Social Comparison,Evaluating oneself negatively by comparing one...,merged,social comparison,1,1,1.666667,60,1.0,All my friends have a gf and they are getting ...,1.7% (N=60)


## 9. Build, review, and save a codebook

Codebook creation is local. Each merged construct becomes a codebook entry with
its definition, sampled example quotes, and an initially empty counterexample
list. Quotes are deduplicated and sampled from the pooled quotes for each group;
they are not balanced across constituents.

Review definitions and examples before coding. Keep list columns as real Python
lists when editing; display strings can split quotes containing semicolons.


In [45]:
codebook = build_codebook(
    merged=final_merged,
    name="Discovery tutorial codebook",
    version=1,
    n_examples=n_codebook_examples,
    seed=seed,
)
codebook_frame = codebook_to_frame(codebook)
display(codebook_frame)


,construct,definition,examples,counter_examples
0,Acceptance,A cognitive and behavioral stance of acknowled...,[We generally accepting and just accept it],[]
1,Activating Event,"External situations, environmental stressors, ...",[I’ve had a lot of trauma and bad things happe...,[]
2,Adaptive Coping Strategies,Positive behavioral and cognitive actions util...,"[Just had to say it, that’s all., How did you ...",[]
3,All-or-Nothing Thinking,A cognitive distortion where situations are pe...,"[Is either doing what they tell me, since its ...",[]
4,Anhedonia and Emotional Numbing,A physiological and emotional state characteri...,"[I feel empty., Nothing is funny nothing is sa...",[]
5,Anxiety,"A high level of emotional distress, worry, and...",[(Not to mention the added anxiety of trying t...,[]
6,Avoidance Behavior,"The behavioral act of staying away from, escap...","[I don't wanna die, I don't want to hurt my lo...",[]
7,Beck's Cognitive Triad,"A pervasive, interconnected negative view of o...",[Fucking hell nothing about life is good now i...,[]
8,Behavioral Activation,"Deliberately engaging in physical activities, ...","[I literally have a great job, I volunteer a t...",[]
9,Behavioral Exposure,Intentionally engaging in novel or anxiety-pro...,"[I went out eating alone for the first time!, ...",[]


In [ ]:
# Make any manual edits to codebook_frame before running this cell.
# Example:
# codebook_frame.at[0, "definition"] = "A clearer definition."
# codebook_frame.at[0, "counter_examples"] = ["A quote that should not count."]

if codebook_frame["definition"].fillna("").str.strip().eq("").any():
    raise ValueError("Fill in missing codebook definitions before saving.")

reviewed_codebook = frame_to_codebook(
    codebook_frame,
    metadata=codebook["metadata"],
)
saved_codebook_path = save_codebook(
    reviewed_codebook,
    codebook_dir / "discovered_codebook.json",
)
print("Use this codebook path in the coding pipeline:", saved_codebook_path.resolve())
